### Problema 1: Detección de scrapers en una ventana de tráfico


Creamos la sesion de spark
Cargamos el dataset e imprimimos el schema

In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MeliChallenge")
    .getOrCreate()
)

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../../data/session_requests.csv")
)

df.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- request_time: timestamp (nullable = true)
 |-- path: string (nullable = true)
 |-- path_type: string (nullable = true)
 |-- d2id: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- user_agent: string (nullable = true)



![Descripción](images/image.png)

Visualizamos el dataframe

In [ ]:
# Vemos las primeras 10 filas del dataframe
df.show(10, truncate=False)

+-----------+-----------------------+--------------------+---------+------------------------------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------+
|session_id |request_time           |path                |path_type|d2id                                |ip_address    |user_agent                                                                                                                             |
+-----------+-----------------------+--------------------+---------+------------------------------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------+
|session_918|2026-06-15 14:00:01.947|/search?q=mesa      |search   |4504399e-5ac7-4fca-8755-7bc35e9f74be|104.161.135.39|Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 

Empezamos con el analisis de los datos

In [12]:
# Vemos el largo del dataframe y la cantidad de valores distintos de cada columna

from pyspark.sql.functions import countDistinct
print(f"Largo dataframe: {df.count()}")
for column in df.columns:
    count = df.select(countDistinct(column)).collect()[0][0]
    print(f"{column}: {count}")

Largo dataframe: 186610
session_id: 1350
request_time: 185242
path: 68168
path_type: 2
d2id: 13773
ip_address: 2264
user_agent: 9


Algunos datos sobre esto:

- Largo dataframe (186,610) vs session_id (1,350):Es una interactividad altísima. Esto significa que los usuarios no entran y se van inmediatamente, sino que pasan tiempo navegando, buscando productos o recorriendo el sitio.

- path (68,168) vs path_type (2): El dataset está concentrado exclusivamente en el descubrimiento y compra

- d2id (13,773) vs session_id (1,350): Hay 10 veces más identificadores de dispositivo (d2id) que sesiones activas.

- ip_address (2,264) vs session_id (1,350): puede que predomine el tráfico en movimiento (móvil) donde los usuarios cambian de Wi-Fi a datos móviles o bien hay presencia de bots/scrapers que rotan IPs constantemente para evadir bloqueos mientras extraen información.



In [ ]:
# Vemos si hay valores nulos: El dataset no contiene valores nulos

from pyspark.sql.functions import col, sum

nulls = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

nulls.show()


+----------+------------+----+---------+----+----------+----------+
|session_id|request_time|path|path_type|d2id|ip_address|user_agent|
+----------+------------+----+---------+----+----------+----------+
|         0|           0|   0|        0|   0|         0|         0|
+----------+------------+----+---------+----+----------+----------+



In [ ]:
# Vemos si hay filas duplicadas: el dataset no contiene filas duplicadas

total_rows = df.count()
distinct_rows = df.distinct().count()

print(f"Total de filas: {total_rows}")
print(f"Filas distintas: {distinct_rows}")
print(f"Duplicados: {total_rows - distinct_rows}")



Total de filas: 186610
Filas distintas: 186610
Duplicados: 0


In [ ]:
from pyspark.sql.functions import min, max, date_format, round
# Vemos la ventana de tiempo que tiene el dataframe
df.select(
    date_format(min("request_time"), "yyyy-MM-dd HH:mm").alias("inicio"),
    date_format(max("request_time"), "yyyy-MM-dd HH:mm").alias("fin"),
    round(((max("request_time").cast("long") - min("request_time").cast("long"))/60/60),2).alias("Diferencia en horas")
).show()

+----------------+----------------+-------------------+
|          inicio|             fin|Diferencia en horas|
+----------------+----------------+-------------------+
|2026-06-15 14:00|2026-06-15 18:18|               4.31|
+----------------+----------------+-------------------+



In [ ]:
# Vemos que porcentaje es de busqueda(37%) y que porcentaje es de item(63%) para entender el comportamiento de los usuarios
# Se puede notar que pasan mas tiempo comparando items que haciendo una busqueda
total = df.count()
(
    df.groupBy("path_type")
    .count()
    .withColumn("percentage (%)", round(col("count") / total * 100,2))
    .show()
)

+---------+------+--------------+
|path_type| count|percentage (%)|
+---------+------+--------------+
|     item|118428|         63.46|
|   search| 68182|         36.54|
+---------+------+--------------+



In [ ]:
from pyspark.sql.functions import regexp_extract

# Ahora si veamos en busquedas, la cantidad de elementos distintos y la cantidad de cada uno de ellos

searches = (
    df
    .filter(df.path_type == "search")
    .withColumn(
        "search_query",
        regexp_extract("path", r"[?&]q=([^&]+)", 1)
    )
)

from pyspark.sql.functions import count, col

# 1. Agrupamos, contamos, ORDENAMOS de mayor a menor y tomamos los primeros 20
busquedas_con_conteo = (searches
                        .groupBy("search_query")
                        .agg(count("*").alias("total"))
                        .orderBy(col("total").desc())  # <-- El ordenamiento va aquí afuera
                        .collect())

# 2. Creamos la lista con el formato "elemento : cantidad"
lista_busquedas = [f"{row['search_query']} : {row['total']}" for row in busquedas_con_conteo]

print(lista_busquedas)
print("Cantidad de elementos distintos:", searches.select("search_query").distinct().count())

# Estos 25 productos tienen una cantidad de elementos cercana.


['parlante : 2811', 'auriculares : 2796', 'cargador : 2789', 'campera : 2785', 'sillon : 2769', 'notebook : 2767', 'bicicleta : 2764', 'silla : 2760', 'televisor : 2756', 'microondas : 2754', 'patineta : 2740', 'cortina : 2735', 'guitarra : 2731', 'cafetera : 2727', 'heladera : 2719', 'teclado : 2713', 'perfume : 2713', 'reloj : 2705', 'ventilador : 2701', 'celular : 2700', 'mouse : 2697', 'mesa : 2664', 'mochila : 2639', 'lampara : 2636', 'zapatillas : 2611']
Cantidad de elementos distintos: 25


In [18]:
from pyspark.sql.functions import regexp_extract, count, col

# Vemos el id de los item, buscando en path y como se repiten en las busquedas
# 1. Filtramos por "item" y extraemos el item_id
items = (
    df
    .filter(df.path_type == "item")
    .withColumn(
        "item_id",
        regexp_extract("path", r"/item/([^/?]+)", 1)
    )
)

# 2. Agrupamos por item_id, contamos, ORDENAMOS de mayor a menor y tomamos los primeros 20
items_con_conteo = (items
                    .groupBy("item_id")
                    .agg(count("*").alias("total"))
                    .orderBy(col("total").desc())
                    .take(50))

# 3. Creamos la lista con el formato "elemento : cantidad"
lista_items = [f"{row['item_id']} : {row['total']}" for row in items_con_conteo]

# 4. Imprimimos los resultados en consola
print(lista_items)
print("Cantidad de elementos distintos:", items.select("item_id").distinct().count())


# Podemos ver que no hay un "Producto Estrella" dominante
# Los 20 productos principales apenas suman unas 166 visitas en conjunto.

['MLA175177 : 11', 'MLA118567 : 10', 'MLA120070 : 9', 'MLA118544 : 9', 'MLA118255 : 9', 'MLA118546 : 9', 'MLA175940 : 9', 'MLA186831 : 8', 'MLA194052 : 8', 'MLA183370 : 8', 'MLA156899 : 8', 'MLA164982 : 8', 'MLA113497 : 8', 'MLA118385 : 8', 'MLA164060 : 8', 'MLA141409 : 8', 'MLA141625 : 7', 'MLA141730 : 7', 'MLA183422 : 7', 'MLA175154 : 7', 'MLA152793 : 7', 'MLA146340 : 7', 'MLA104121 : 7', 'MLA152845 : 7', 'MLA138452 : 7', 'MLA193361 : 7', 'MLA166176 : 7', 'MLA141469 : 7', 'MLA118518 : 7', 'MLA165972 : 7', 'MLA153253 : 7', 'MLA118336 : 7', 'MLA152676 : 7', 'MLA141845 : 7', 'MLA175101 : 7', 'MLA150469 : 7', 'MLA152691 : 7', 'MLA175520 : 7', 'MLA157199 : 7', 'MLA110053 : 7', 'MLA119527 : 7', 'MLA127773 : 7', 'MLA141585 : 7', 'MLA175279 : 7', 'MLA111428 : 7', 'MLA153793 : 7', 'MLA183310 : 7', 'MLA141771 : 7', 'MLA183607 : 7', 'MLA152979 : 7']
Cantidad de elementos distintos: 68143


In [ ]:
# La cantidad de ips diferentes por cada sesion

session_ips = (
    df
    .groupBy("session_id")
    .agg(
        countDistinct("ip_address").alias("unique_ips")
    )
    .orderBy("unique_ips", ascending=False)
)

session_ips.show(20)

## Aca ya notamos un comportamiento anomalo: Es en principio sospechoso que haya pocas sesiones con tal cantidad de ips.

+------------+----------+
|  session_id|unique_ips|
+------------+----------+
|session_1340|       138|
|session_1336|       137|
|session_1332|       126|
|session_1335|       112|
|session_1339|        88|
|session_1331|        74|
|session_1337|        74|
|session_1338|        70|
|session_1334|        54|
|session_1333|        51|
|session_1253|         1|
|session_1011|         1|
| session_564|         1|
| session_808|         1|
| session_339|         1|
|session_1322|         1|
| session_708|         1|
| session_237|         1|
| session_834|         1|
| session_885|         1|
+------------+----------+
only showing top 20 rows


In [ ]:
# La cantidad de sesiones diferentes por cada ip

ip_sessions = (
    df
    .groupBy("ip_address")
    .agg(
        countDistinct("session_id").alias("unique_sessions")
    )
    .orderBy("unique_sessions", ascending=False)
)

ip_sessions.show(20)

# Toda ip solo forma parte de una sesion.

+---------------+---------------+
|     ip_address|unique_sessions|
+---------------+---------------+
|  137.98.245.82|              1|
|121.229.250.133|              1|
|    82.87.82.19|              1|
| 79.248.223.134|              1|
| 132.177.198.81|              1|
| 179.232.70.224|              1|
|  113.95.222.41|              1|
|196.205.170.127|              1|
|  218.53.55.138|              1|
|  98.206.44.232|              1|
|  187.84.245.33|              1|
|  21.253.202.35|              1|
| 172.145.133.76|              1|
| 43.221.169.221|              1|
| 139.177.203.57|              1|
| 201.11.175.236|              1|
|  65.240.99.253|              1|
| 73.124.146.240|              1|
| 167.197.34.245|              1|
|  76.35.186.222|              1|
+---------------+---------------+
only showing top 20 rows


In [ ]:
# La cantidad de dispositivos por cada sesion

session_devices = (
    df
    .groupBy("session_id")
    .agg(
        countDistinct("d2id").alias("unique_devices")
    )
    .orderBy("unique_devices", ascending=False)
)

session_devices.show(10)
session_devices.orderBy("unique_devices",asc=False).show(10)
# Aca podriamos ver los percentiles
from pyspark.sql.functions import expr, percentile_approx

# Calculamos los percentiles sobre la columna 'unique_devices'
session_devices.select(
    percentile_approx("unique_devices", 0.25).alias("p25"),
    percentile_approx("unique_devices", 0.50).alias("p50_mediana"),
    percentile_approx("unique_devices", 0.75).alias("p75"),
    percentile_approx("unique_devices", 0.90).alias("p90"),
    percentile_approx("unique_devices", 0.99).alias("p99")
).show()

# Aca notamos tambien que la cantidad de dispositivos por sesion tambien tiene un comportamiento extraño

+------------+--------------+
|  session_id|unique_devices|
+------------+--------------+
|session_1302|          1167|
|session_1305|          1128|
|session_1309|          1097|
|session_1303|          1066|
|session_1306|           909|
|session_1311|           904|
|session_1304|           760|
|session_1314|           684|
|session_1315|           677|
|session_1312|           660|
+------------+--------------+
only showing top 10 rows
+------------+--------------+
|  session_id|unique_devices|
+------------+--------------+
| session_244|             1|
| session_463|             1|
| session_414|             1|
|session_1253|             1|
|  session_10|             1|
|session_1011|             1|
| session_211|             1|
| session_708|             1|
| session_689|             1|
|session_1083|             1|
+------------+--------------+
only showing top 10 rows
+---+-----------+---+---+---+
|p25|p50_mediana|p75|p90|p99|
+---+-----------+---+---+---+
|  1|          1|  1

In [ ]:
# Vemos la cantidad de requests que hay por sesion, y hacemos un summary para detectar si hay comportamientos extraños.

from pyspark.sql.functions import count


requests_per_session = (
    df.groupBy("session_id")
    .agg(
        count("*").alias("request_count")
    ).orderBy("request_count", ascending=False)
    
)
requests_per_session.show(10)
print("\n")
requests_per_session.select("request_count").summary().show()

NameError: name 'df' is not defined